In [1]:
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from matplotlib.lines import Line2D
from typing import Literal
from sklearn.decomposition import PCA

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats 

from support import prepare_count_matrix

In [21]:
METADATA_PATH = f"{os.getcwd()}/../../hgsoc_data_detective/"
DATA_PATH = f"{os.getcwd()}/../../alignment_results/"
GTF_PATH = f"{os.getcwd()}/../../../../Downloads/refdata-gex-GRCh38-2024-A/genes/genes.gtf"

In [3]:
df = pd.read_excel(METADATA_PATH + "samples_overview.xlsx", header=1, index_col=0)

In [4]:
df = df[["0509", "0626", "Site of origin"]]

In [5]:
df.drop(index=["Total across columns", 2205], inplace=True)

In [6]:
df.value_counts("Site of origin")

Site of origin
Unknown                   17
Omentum                   12
Ovary                      5
Para-aortic lymph node     1
Para-colic gutter          1
Name: count, dtype: int64

In [7]:
df = df.sort_index().sort_values("0509")

In [8]:
len(df)

36

In [9]:
sub_df = df[df["0509"]=="X"]
index_1 = ["230509/" + str(x) for x in sub_df.index]

sub_df = df[df["0626"]=="X"]
index_2 = ["230626/" + str(x) for x in sub_df.index]

In [10]:
len(index_1) + len(index_2)

36

In [11]:
df["Long Sample ID"] = sorted(index_1) + sorted(index_2)

In [12]:
df = df.drop(columns=["0509", "0626"])

In [13]:
df.head()

,Site of origin,Long Sample ID
Sample #,,
2018,Omentum,230509/2018
2023,Unknown,230509/2023
2094,Unknown,230509/2094
2126,Unknown,230509/2126
2129,Ovary,230509/2129


In [14]:
omentum_samples = list(df[df["Site of origin"]=="Omentum"].index)
omentum_samples = [str(x) for x in omentum_samples]
ovary_samples = list(df[df["Site of origin"]=="Ovary"].index)
ovary_samples = [str(x) for x in ovary_samples]

In [38]:
count_matrix = prepare_count_matrix(DATA_PATH)
# Select only bulk samples
count_matrix = count_matrix[~count_matrix.index.str.contains("2304")]
# Only consider genes that have more than 10 read counts in total
count_matrix = count_matrix[count_matrix.columns[count_matrix.sum(axis=0) >= 10]]
# Add site of origin 
count_matrix.index.name = "Long Sample ID"
count_df = pd.merge(df, count_matrix, how="inner", left_on="Long Sample ID", right_on="Long Sample ID")
count_df = count_df.set_index("Long Sample ID")

In [59]:
count_df = count_df[(count_df["Site of origin"]=="Omentum")|(count_df["Site of origin"]=="Ovary")]

In [60]:
count_df.head()

,Site of origin,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,...,ENSG00000291299,ENSG00000291300,ENSG00000291309,ENSG00000291317,ENSG00000291326,ENSG00000291342,ENSG00000292223,ENSG00000292246,ENSG00000292271,ENSG00000292309
Long Sample ID,,,,,,,,,,,,,,,,,,,,,
230509/2018,Omentum,205,2,1629,81,78,615,1159,457,443,...,362,21,6,50,7,0,0,1,1,2
230509/2129,Ovary,221,9,600,84,89,148,1821,233,129,...,132,35,4,60,2,0,0,2,0,0
230509/2186,Omentum,215,6,637,69,52,731,1579,276,941,...,205,12,1,70,3,0,1,0,0,0
230509/2202,Omentum,80,7,1134,40,33,634,2251,344,655,...,178,5,2,6,10,0,2,0,0,9
230509/2444,Omentum,598,8,796,98,82,204,1460,281,224,...,203,21,1,7,2,0,1,2,0,5


In [40]:
len(count_df)

36

In [41]:
# Get mapping from ensemble gene ids to gene names
# I originally downloaded a reference genome for alignment, and this file was part of that download
with open(GTF_PATH) as f:
    gtf = list(f)
    gtf = gtf[5:]

gtf = [x for x in gtf if 'gene_id "' in x and 'gene_name "' in x]

gtf = list(map(lambda x: (x.split('gene_id "')[1].split('"')[0], x.split('gene_name "')[1].split('"')[0]), gtf))
gtf = list(set(gtf))

In [42]:
gtf[:5]

[('ENSG00000289321', 'ENSG00000289321'),
 ('ENSG00000231249', 'ITPR1-DT'),
 ('ENSG00000197409', 'H3C4'),
 ('ENSG00000248367', 'ENSG00000248367'),
 ('ENSG00000100722', 'ZC3H14')]

In [61]:
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=count_df.drop(columns=["Site of origin"]),
    metadata=count_df[["Site of origin", "ENSG00000000003"]],
    design_factors="Site of origin",
    refit_cooks=True,
    inference=inference,
)
dds.deseq2()

stat_res = DeseqStats(dds, inference=inference)
stat_res.summary()
res = stat_res.results_df.rename(index=dict(gtf))

# Set some threshold to only retain genes that are signficantly DE
final_df = res[(
    (res["padj"] < 0.05) & 
    (abs(res["log2FoldChange"]) > 0.5) & 
    (res["baseMean"] > 10)
)]

Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 1.79 seconds.

Fitting dispersion trend curve...
... done in 0.22 seconds.

Fitting MAP dispersions...
... done in 2.07 seconds.

Fitting LFCs...
... done in 1.33 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 1233 outlier genes.

Fitting dispersions...
... done in 0.08 seconds.

Fitting MAP dispersions...
... done in 0.09 seconds.

Fitting LFCs...
... done in 0.09 seconds.

Running Wald tests...


Log2 fold change & Wald test p-value: Site of origin Ovary vs Omentum
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
ENSG00000000003   366.756933        0.077917  0.467364  0.166716  0.867594   
ENSG00000000005     8.359633        0.405498  0.648817  0.624980  0.531984   
ENSG00000000419  1491.226230        0.446539  0.320584  1.392894  0.163652   
ENSG00000000457   149.612318       -0.208430  0.398193 -0.523440  0.600668   
ENSG00000000460   118.481984       -0.062807  0.408845 -0.153620  0.877909   
...                      ...             ...       ...       ...       ...   
ENSG00000291342     1.059014        0.109792  1.870265  0.058704  0.953188   
ENSG00000292223     1.557657       -0.062063  1.113681 -0.055728  0.955559   
ENSG00000292246     1.006060        1.226760  1.187685  1.032899  0.301651   
ENSG00000292271     0.337331       -0.667889  2.769013 -0.241201  0.809399   
ENSG00000292309     3.141928       -0.945252  1.235006 -0.765382  0.4440

... done in 0.50 seconds.



In [62]:
final_df.sort_values(by="log2FoldChange", key=abs, ascending=False).head(10)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
STAR,1042.375870,6.308016,0.740325,8.520605,1.587225e-17,3.676649e-13
LINC01391,33.064149,5.934592,0.825984,7.184871,6.727066e-13,7.791288e-09
PCDH11X,77.668997,5.875874,0.883439,6.651137,2.908378e-11,2.245655e-07
WIF1,17.268680,4.957428,0.985373,5.031016,4.878862e-07,6.278554e-04
IRS4,36.983801,4.783443,0.932906,5.127462,2.936737e-07,5.232813e-04
ENSG00000248935,16.328000,-4.774394,1.048520,-4.553460,5.277060e-06,4.365637e-03
PTPRT,547.900619,-4.572006,1.013815,-4.509706,6.491745e-06,4.556811e-03
KRTAP2-3,43.573226,-4.324028,0.937852,-4.610565,4.015758e-06,3.577731e-03
ADGRG4,14.879860,4.322211,0.851216,5.077688,3.820559e-07,6.278554e-04
ENPP6,18.278811,4.008606,0.694570,5.771348,7.863979e-09,2.602303e-05


In [77]:
res[res.index.str.contains("ADIPOQ")]

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
ADIPOQ,4.451354,-1.079181,1.595646,-0.676329,0.498832,NaN
ADIPOQ-AS1,0.381142,1.368227,2.485712,0.550437,0.582020,NaN
